# Movement Phase Analysis


## Imports And Configuration

In [ ]:
%matplotlib inline
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt

notebook_dir = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

from manual_calibrated_er_mr_lr import load_mat_raw
from movement_phase_helpers import (
    assign_phase_to_stimuli,
    compute_movement_phase,
    detect_cycle_boundaries,
    extract_movement_signal,
    make_emg_envelope,
    validate_filter_parameters,
)

n_phase_bins = 8
phase_bin_labels = np.arange(1, n_phase_bins + 1)
phase_cmap = "tab10"
component_colors = {"er": "tab:green", "mr": "tab:orange", "lr": "tab:blue"}
response_prefix = {"er": "early", "mr": "middle", "lr": "late"}
response_label_order = ["early", "middle", "late"]

available_response_metric_names = [
    "p2p_amplitude",
    "absolute_peak_amplitude",
    "signed_peak_amplitude",
    "prominence",
    "auc_abs",
    "rms",
    "start_latency_ms",
    "peak_latency_ms",
    "end_latency_ms",
    "duration_ms",
    "amplitude_noise_ratio",
    "p2p_noise_ratio",
]

m_response_metrics = [
    ("middle_p2p_amplitude", "M-response p2p amplitude"),
    ("middle_rms", "M-response RMS"),
    ("middle_peak_latency_ms", "M-response peak latency"),
]

lr_response_metrics = [
    ("late_p2p_amplitude", "LR p2p amplitude"),
    ("late_rms", "LR RMS"),
    ("late_auc_abs", "LR absolute AUC"),
    ("late_peak_latency_ms", "LR peak latency"),
]

lr_vs_m_metrics = [
    ("late_p2p_amplitude", "LR p2p amplitude"),
    ("late_rms", "LR RMS"),
    ("late_auc_abs", "LR absolute AUC"),
]

latency_outlier_mad_k = 3.0
cycle_duration_flag_range_s = (0.25, 2.0)

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

movement_phase_config = {
    "channel": "GM Right",
    "emg_band": (30.0, 500.0),
    "envelope_lowpass": 5.0,
    "movement_band": (0.5, 2.5),
    "filter_order": 4,
    "qc_window_s": (100.0, 110.0),
}

mat_path = project_root / "data" / "raw" / "pig_4_9_14_day_after_SCI.mat"
manual_annotation_path = project_root / "notebooks" / "annotations" / "manual_er_mr_lr-annot.fif"
reviewed_metrics_path = project_root / "outputs" / "metrics" / "manual_calibrated_candidates_after_review.csv"
final_annotation_path = project_root / "outputs" / "annotations" / "stimulus_er_mr_lr_manual_calibrated_auto-annot.fif"

pd.set_option("display.max_columns", 160)


## Helper Functions: Movement Phase

In [ ]:
def emg_envelope(x, sfreq, band=(30, 500), lowpass=5.0):
    """Create a rectified low-pass EMG envelope from raw EMG.

    Input shape is one signal `(n_times,)` or channels by time. Filtering is
    applied along the last axis. Output has the same shape as the input.
    """
    nyq = sfreq / 2
    sos_bp = butter(4, [band[0] / nyq, band[1] / nyq], "band", output="sos")
    sos_lp = butter(4, lowpass / nyq, "low", output="sos")
    x_bp = sosfiltfilt(sos_bp, x, axis=-1)
    return sosfiltfilt(sos_lp, np.abs(x_bp), axis=-1)


def make_phase_bins(n_phase_bins):
    """Return equal phase-bin edges and centers over `-pi..+pi`.

    Input is the requested number of bins. Outputs are arrays with shapes
    `(n_phase_bins + 1,)` for edges and `(n_phase_bins,)` for centers.
    """
    edges = np.linspace(-np.pi, np.pi, int(n_phase_bins) + 1)
    centers = (edges[:-1] + edges[1:]) / 2
    return edges, centers


def assign_phase_bins(stim_phases, phase_bin_edges):
    """Assign one human-readable phase bin to each stimulus.

    Input is a dataframe with one row per stimulus and a `movement_phase` column.
    Output is a copy with `phase_bin`, `phase_bin_center`, `phase_sin`, and
    `phase_cos`. Missing or invalid phases remain missing.
    """
    result = stim_phases.copy()
    n_bins = len(phase_bin_edges) - 1
    phase = pd.to_numeric(result["movement_phase"], errors="coerce").to_numpy(dtype=float)
    valid = np.isfinite(phase)

    adjusted = phase.copy()
    adjusted[valid & np.isclose(adjusted, np.pi)] = np.nextafter(np.pi, -np.inf)
    bins = np.full(len(result), np.nan)
    bins[valid] = np.digitize(adjusted[valid], phase_bin_edges, right=False)
    valid_bins = valid & (bins >= 1) & (bins <= n_bins)

    result["phase_bin"] = pd.Series(pd.NA, index=result.index, dtype="Int64")
    result.loc[valid_bins, "phase_bin"] = bins[valid_bins].astype(int)
    centers = pd.Series({bin_id: center for bin_id, center in zip(range(1, n_bins + 1), (phase_bin_edges[:-1] + phase_bin_edges[1:]) / 2)})
    result["phase_bin_center"] = result["phase_bin"].map(centers).astype(float)
    result["phase_sin"] = np.sin(result["movement_phase"])
    result["phase_cos"] = np.cos(result["movement_phase"])
    return result


def validate_phase_alignment(movement_phase, stim_samples, raw, sfreq):
    """Check that stimulus samples can index the continuous movement phase.

    Inputs are the continuous phase vector, integer stimulus samples, the Raw
    object, and sampling frequency. Output is a one-row dataframe with alignment
    counts and bounds.
    """
    stim_samples = np.asarray(stim_samples)
    stim_indices = stim_samples - raw.first_samp
    integer_samples = np.issubdtype(stim_samples.dtype, np.integer)
    within_bounds = (stim_indices >= 0) & (stim_indices < len(movement_phase))
    finite_phase = np.isfinite(movement_phase[stim_indices[within_bounds]]) if within_bounds.any() else np.array([], dtype=bool)

    summary = pd.DataFrame([{
        "movement_phase_samples": len(movement_phase),
        "raw_samples": raw.n_times,
        "one_phase_per_raw_sample": len(movement_phase) == raw.n_times,
        "stimuli_total": len(stim_samples),
        "stimulus_samples_integer": bool(integer_samples),
        "stimuli_within_phase_bounds": int(within_bounds.sum()),
        "stimuli_out_of_bounds": int((~within_bounds).sum()),
        "stimuli_with_finite_phase": int(finite_phase.sum()),
        "raw_first_samp": int(raw.first_samp),
        "sfreq": float(sfreq),
    }])

    if not bool(summary.loc[0, "one_phase_per_raw_sample"]):
        raise ValueError("movement_phase does not have one value per raw sample")
    if not integer_samples:
        raise TypeError("stim_samples must be integer sample indices")
    if not within_bounds.all():
        raise ValueError("Some stimulus samples are outside movement_phase bounds")
    if finite_phase.sum() != len(stim_samples):
        raise ValueError("Some stimulus samples have missing or non-finite movement phase")
    return summary


## Helper Functions: Response Tables

In [ ]:
def _first_response_per_stimulus(reviewed_metrics, label):
    """Return the first reviewed response of one label per stimulus.

    Input is the long reviewed metrics table. Output has at most one row per
    `stimulus_index`, preserving existing metric definitions.
    """
    subset = reviewed_metrics.loc[reviewed_metrics["label"].eq(label)].copy()
    if subset.empty:
        return subset
    sort_cols = ["stimulus_index", "component_rank", "peak_latency_ms"]
    sort_cols = [col for col in sort_cols if col in subset.columns]
    return subset.sort_values(sort_cols).groupby("stimulus_index", as_index=False).first()


def _strongest_lr_per_stimulus(reviewed_metrics):
    """Return the strongest reviewed LR row per stimulus.

    Input is the long reviewed metrics table. Output has at most one LR row per
    `stimulus_index`, ranked by prominence then absolute amplitude.
    """
    subset = reviewed_metrics.loc[reviewed_metrics["label"].eq("lr")].copy()
    if subset.empty:
        return subset
    sort_cols = ["stimulus_index", "prominence", "absolute_peak_amplitude", "peak_latency_ms"]
    ascending = [True, False, False, True]
    return subset.sort_values(sort_cols, ascending=ascending).groupby("stimulus_index", as_index=False).first()


def build_response_by_stim(stim_phases, reviewed_metrics, metric_columns):
    """Create one row per stimulation with phase and reviewed ER/MR/LR metrics.

    Inputs are the per-stimulus phase table and long reviewed metrics table.
    Output is `response_by_stim`, one row per stimulus. Metrics are NaN when a
    reviewed response is not available for that stimulus.
    """
    base = stim_phases.rename(columns={"stimulus_time": "stimulus_time_s"}).copy()
    base.insert(0, "stimulus_id", base["stimulus_index"].astype(int))
    response_by_stim = base.copy()

    for label, prefix in response_prefix.items():
        component = _strongest_lr_per_stimulus(reviewed_metrics) if label == "lr" else _first_response_per_stimulus(reviewed_metrics, label)
        keep = ["stimulus_index", *[col for col in metric_columns if col in component.columns]]
        keep += [col for col in ["start_s", "end_s", "annotation_onset_s", "component_rank"] if col in component.columns]
        component = component.loc[:, list(dict.fromkeys(keep))].copy() if len(component) else pd.DataFrame(columns=keep)
        component = component.rename(columns={col: f"{prefix}_{col}" for col in component.columns if col != "stimulus_index"})
        response_by_stim = response_by_stim.merge(component, on="stimulus_index", how="left", validate="one_to_one")
        present_col = f"{prefix}_peak_latency_ms"
        response_by_stim[f"{prefix}_present"] = response_by_stim[present_col].notna() if present_col in response_by_stim else False

    counts = reviewed_metrics.pivot_table(index="stimulus_index", columns="label", values="peak_latency_ms", aggfunc="size", fill_value=0)
    for label, prefix in response_prefix.items():
        label_counts = counts[label] if label in counts else pd.Series(dtype=int)
        response_by_stim[f"{prefix}_count"] = response_by_stim["stimulus_index"].map(label_counts).fillna(0).astype(int)

    response_by_stim["phase_at_er"] = response_by_stim["movement_phase"].where(response_by_stim["early_present"])
    response_by_stim["phase_at_mr"] = response_by_stim["movement_phase"].where(response_by_stim["middle_present"])
    response_by_stim["phase_at_lr"] = response_by_stim["movement_phase"].where(response_by_stim["late_present"])
    return response_by_stim


def summarize_by_phase_bin(response_by_stim, metric_columns, n_phase_bins):
    """Summarize per-stimulus response metrics by phase bin.

    Input is `response_by_stim`. Output is one row per phase bin with stimulus
    counts, descriptive metric statistics, and LR occurrence fraction.
    """
    bin_labels = np.arange(1, int(n_phase_bins) + 1)
    grouped = response_by_stim.groupby("phase_bin", observed=False)
    summary = grouped.size().rename("total_stim_count").reindex(bin_labels, fill_value=0).to_frame()

    continuous_cols = []
    for prefix in response_label_order:
        for metric in metric_columns:
            col = f"{prefix}_{metric}"
            if col in response_by_stim.columns:
                continuous_cols.append(col)

    if continuous_cols:
        stats = grouped[continuous_cols].agg(["count", "mean", "median", "std"]).reindex(bin_labels)
        stats.columns = [f"{col}_{stat}" for col, stat in stats.columns]
        summary = summary.join(stats)

    late_count = grouped["late_present"].sum().reindex(bin_labels, fill_value=0).astype(int)
    summary["late_response_count"] = late_count
    summary["late_response_fraction"] = summary["late_response_count"] / summary["total_stim_count"].replace(0, np.nan)
    return summary.reset_index().rename(columns={"phase_bin": "phase_bin"})


def add_lr_latency_outliers(response_by_stim, value_col="late_peak_latency_ms", mad_k=3.0):
    """Mark robust outliers in LR latency without excluding data.

    Input is `response_by_stim`. Output is a copy with robust latency center,
    robust z-score, and boolean outlier columns.
    """
    result = response_by_stim.copy()
    values = pd.to_numeric(result[value_col], errors="coerce")
    center = float(values.median())
    mad = float((values - center).abs().median())
    scaled_mad = 1.4826 * mad if np.isfinite(mad) else np.nan
    if not np.isfinite(scaled_mad) or scaled_mad == 0:
        result["late_latency_robust_z"] = np.nan
        result["late_latency_outlier"] = False
        return result, center, scaled_mad

    robust_z = (values - center) / scaled_mad
    result["late_latency_robust_z"] = robust_z
    result["late_latency_outlier"] = robust_z.abs() > mad_k
    return result, center, scaled_mad


## Helper Functions: Plotting

In [ ]:
def _phase_scatter(ax, data, x_col, y_col, title, xlabel, ylabel):
    """Draw a phase-colored scatter plot for one x/y metric pair."""
    plot_df = data.dropna(subset=[x_col, y_col, "phase_bin"]).copy()
    if plot_df.empty:
        ax.set_title(f"{title} (no data)")
        ax.axis("off")
        return None
    scatter = ax.scatter(
        plot_df[x_col], plot_df[y_col],
        c=plot_df["phase_bin"].astype(float),
        cmap=phase_cmap,
        vmin=1,
        vmax=n_phase_bins,
        s=36,
        alpha=0.75,
        edgecolor="white",
        linewidth=0.4,
    )
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)
    return scatter


def plot_raw_envelope_phase_segment(time_s, raw_emg, envelope, movement_signal, movement_phase, response_by_stim, cycle_boundaries, sfreq, start_s, stop_s):
    """Plot raw EMG, envelope/movement signal, phase, and stimulus markers.

    Inputs are continuous arrays plus `response_by_stim`. Output is a stacked
    Matplotlib figure for one time segment.
    """
    mask = (time_s >= start_s) & (time_s <= stop_s)
    if mask.sum() < 2:
        raise ValueError(f"QC window {start_s}..{stop_s} s does not overlap the recording")

    fig, axes = plt.subplots(4, 1, figsize=(15, 7), sharex=True, height_ratios=[1, 1, 1, 1])
    axes[0].plot(time_s[mask], raw_emg[mask], color="0.60", linewidth=0.7)
    axes[0].set_ylabel("raw GM R", rotation=0, ha="right", va="center")
    axes[1].plot(time_s[mask], envelope[mask], color="tab:blue", linewidth=1.1)
    axes[1].set_ylabel("envelope", rotation=0, ha="right", va="center")
    axes[2].plot(time_s[mask], movement_signal[mask], color="tab:cyan", linewidth=1.0)
    axes[2].set_ylabel("movement", rotation=0, ha="right", va="center")
    axes[3].plot(time_s[mask], movement_phase[mask], color="tab:purple", linewidth=1.0)
    axes[3].set_ylabel("phase", rotation=0, ha="right", va="center")
    axes[3].set_yticks([-np.pi, 0, np.pi])
    axes[3].set_yticklabels(["-pi", "0", "+pi"])

    boundary_times = cycle_boundaries / sfreq
    for boundary_s in boundary_times[(boundary_times >= start_s) & (boundary_times <= stop_s)]:
        axes[2].axvline(boundary_s, color="0.2", alpha=0.25, linewidth=0.8)
        axes[3].axvline(boundary_s, color="0.2", alpha=0.25, linewidth=0.8)

    qc_stims = response_by_stim.loc[response_by_stim["stimulus_time_s"].between(start_s, stop_s)]
    cmap = plt.get_cmap(phase_cmap, n_phase_bins)
    for row in qc_stims.itertuples():
        color = cmap((int(row.phase_bin) - 1) % n_phase_bins) if pd.notna(row.phase_bin) else "tab:red"
        for ax in axes:
            ax.axvline(row.stimulus_time_s, color=color, alpha=0.55, linewidth=0.9)
        axes[3].text(row.stimulus_time_s, np.pi, str(row.phase_bin), ha="center", va="bottom", fontsize=7, color=color)

    for ax in axes:
        ax.grid(alpha=0.18)
        ax.spines[["top", "right"]].set_visible(False)
    axes[-1].set_xlabel("Time, s")
    fig.suptitle(f"Raw EMG, envelope, movement signal, and phase: {start_s:g}-{stop_s:g} s")
    plt.tight_layout()
    return fig, axes


def plot_lr_vs_m_response(response_by_stim):
    """Plot LR amplitude/energy metrics against M-response amplitude.

    Input is `response_by_stim`. Output is a compact figure with phase-colored
    scatter plots.
    """
    fig, axes = plt.subplots(1, len(lr_vs_m_metrics), figsize=(5.2 * len(lr_vs_m_metrics), 4), squeeze=False)
    scatter = None
    for ax, (y_col, y_label) in zip(axes[0], lr_vs_m_metrics):
        scatter = _phase_scatter(
            ax,
            response_by_stim,
            "middle_p2p_amplitude",
            y_col,
            y_label,
            "M-response p2p amplitude",
            y_label,
        )
    if scatter is not None:
        cbar = fig.colorbar(scatter, ax=axes.ravel().tolist(), shrink=0.85)
        cbar.set_label("Phase bin")
        cbar.set_ticks(phase_bin_labels)
    fig.suptitle("Late response vs M-response, colored by movement phase")
    plt.tight_layout()
    return fig, axes


def plot_lr_latency_outliers(response_by_stim, latency_center_ms, latency_scale_ms, mad_k=3.0):
    """Plot LR latency against M-response and highlight robust outliers.

    Input is `response_by_stim` with `late_latency_outlier`. Output is a
    Matplotlib figure and the dataframe of plotted LR rows.
    """
    plot_df = response_by_stim.dropna(subset=["middle_p2p_amplitude", "late_peak_latency_ms", "phase_bin"]).copy()
    fig, ax = plt.subplots(figsize=(7, 4.5))
    scatter = _phase_scatter(
        ax,
        plot_df,
        "middle_p2p_amplitude",
        "late_peak_latency_ms",
        "LR latency vs M-response",
        "M-response p2p amplitude",
        "LR peak latency, ms",
    )
    outliers = plot_df.loc[plot_df["late_latency_outlier"]]
    if len(outliers):
        ax.scatter(
            outliers["middle_p2p_amplitude"],
            outliers["late_peak_latency_ms"],
            facecolors="none",
            edgecolors="red",
            linewidths=1.8,
            s=90,
            label="robust latency outlier",
        )
        for row in outliers.itertuples():
            ax.annotate(str(row.stimulus_index), (row.middle_p2p_amplitude, row.late_peak_latency_ms), xytext=(4, 4), textcoords="offset points", fontsize=8)
    if np.isfinite(latency_scale_ms) and latency_scale_ms > 0:
        ax.axhline(latency_center_ms, color="black", linewidth=1.0, label="latency median")
        ax.axhspan(latency_center_ms - mad_k * latency_scale_ms, latency_center_ms + mad_k * latency_scale_ms, color="0.7", alpha=0.18, label="median +/- 3 scaled MAD")
    if scatter is not None:
        cbar = fig.colorbar(scatter, ax=ax)
        cbar.set_label("Phase bin")
        cbar.set_ticks(phase_bin_labels)
    ax.legend(loc="best")
    plt.tight_layout()
    return fig, ax, plot_df


def plot_phase_bin_summary(response_by_stim, response_by_phase_bin):
    """Plot M-response, LR strength, LR fraction, and stimulus count by phase bin.

    Inputs are the per-stimulus and per-bin tables. Output is a four-row figure.
    """
    fig, axes = plt.subplots(4, 1, figsize=(9, 9), sharex=True, height_ratios=[2, 2, 1.5, 1])
    plot_specs = [
        ("middle_p2p_amplitude", "M-response p2p amplitude", axes[0]),
        ("late_p2p_amplitude", "LR p2p amplitude", axes[1]),
    ]
    rng = np.random.default_rng(7)
    for col, ylabel, ax in plot_specs:
        values_df = response_by_stim.dropna(subset=["phase_bin", col]).copy()
        if len(values_df):
            jitter = rng.uniform(-0.08, 0.08, len(values_df))
            ax.scatter(values_df["phase_bin"].astype(float) + jitter, values_df[col], s=24, alpha=0.55, color="0.35")
            med = values_df.groupby("phase_bin")[col].median().reindex(phase_bin_labels)
            ax.plot(phase_bin_labels, med, color="tab:red", marker="o", linewidth=1.5, label="median")
            ax.legend(loc="best")
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.25)

    axes[2].plot(response_by_phase_bin["phase_bin"], response_by_phase_bin["late_response_fraction"], color="tab:blue", marker="o", linewidth=1.5)
    axes[2].set_ylabel("LR fraction")
    axes[2].set_ylim(0, max(0.05, np.nanmax(response_by_phase_bin["late_response_fraction"]) * 1.2))
    axes[2].grid(alpha=0.25)

    axes[3].bar(response_by_phase_bin["phase_bin"], response_by_phase_bin["total_stim_count"], color="0.55")
    axes[3].set_ylabel("stimuli")
    axes[3].set_xlabel("Movement phase bin")
    axes[3].set_xticks(phase_bin_labels)
    axes[3].grid(axis="y", alpha=0.25)

    fig.suptitle("M-response, LR, and sample count by movement phase")
    plt.tight_layout()
    return fig, axes


def plot_phase_at_response_counts(response_by_stim):
    """Plot phase-bin counts for stimuli where ER, MR, or LR is present.

    Input is `response_by_stim`. Output is a three-panel bar plot.
    """
    specs = [
        ("early_present", "phase_at_er", "ER"),
        ("middle_present", "phase_at_mr", "MR"),
        ("late_present", "phase_at_lr", "LR"),
    ]
    count_tables = []
    for present_col, phase_col, label in specs:
        counts = (
            response_by_stim.loc[response_by_stim[present_col], "phase_bin"]
            .value_counts()
            .reindex(phase_bin_labels, fill_value=0)
            .sort_index()
        )
        count_tables.append((label, counts))
    ymax = max([counts.max() for _, counts in count_tables] + [1])

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
    for ax, (label, counts) in zip(axes, count_tables):
        ax.bar(counts.index, counts.values, color=component_colors[label.lower()])
        ax.set_title(label)
        ax.set_xlabel("Phase bin")
        ax.set_xticks(phase_bin_labels)
        ax.set_ylim(0, ymax * 1.15)
        ax.grid(axis="y", alpha=0.25)
    axes[0].set_ylabel("Stimuli with response")
    fig.suptitle("phase_at_er / phase_at_mr / phase_at_lr by movement phase bin")
    plt.tight_layout()
    return fig, axes


def plot_10_epochs_by_phase(response_by_stim, phase_response_metrics, raw_emg, time_s, sfreq, n_plots=10, prefer_latency_outliers=True):
    """Plot up to ten LR-containing epochs with phase and metric annotations.

    Inputs are per-stimulus metrics, long response metrics, continuous EMG,
    time axis, and sampling frequency. Output is a figure plus plotted rows.
    """
    rank_df = response_by_stim.loc[response_by_stim["late_present"]].copy()
    if rank_df.empty:
        return None, rank_df
    rank_df["latency_rank"] = rank_df["late_latency_outlier"].astype(int) if "late_latency_outlier" in rank_df else 0
    sort_cols = ["latency_rank", "late_prominence", "late_absolute_peak_amplitude"]
    sort_cols = [col for col in sort_cols if col in rank_df.columns]
    rank_df = rank_df.sort_values(sort_cols, ascending=[False] * len(sort_cols)).head(n_plots)

    fig, axes = plt.subplots(len(rank_df), 1, figsize=(10, max(2.0 * len(rank_df), 4)), sharex=True, squeeze=False)
    axes = axes[:, 0]
    for plot_i, (_, stim_row) in enumerate(rank_df.iterrows()):
        ax = axes[plot_i]
        stim_s = float(stim_row["stimulus_time_s"])
        stim_idx = int(stim_row["stimulus_index"])
        rows = phase_response_metrics.loc[phase_response_metrics["stimulus_index"].eq(stim_idx)].sort_values(["start_s", "label"])
        right_ms = max(70.0, float(rows["end_latency_ms"].max()) + 8.0 if len(rows) else 70.0)
        left_s = max(0.0, stim_s - 0.005)
        right_s = min(time_s[-1], stim_s + right_ms / 1000.0)
        mask = (time_s >= left_s) & (time_s <= right_s)
        x_ms = (time_s[mask] - stim_s) * 1000
        ax.plot(x_ms, raw_emg[mask], color="black", linewidth=0.9)
        ax.axvline(0, color="black", linestyle=":", linewidth=0.8)
        for response in rows.itertuples():
            color = component_colors.get(str(response.label), "0.5")
            label = str(response.label).upper() if plot_i == 0 else None
            ax.axvspan(float(response.start_latency_ms), float(response.end_latency_ms), color=color, alpha=0.18, label=label)
            ax.axvline(float(response.peak_latency_ms), color=color, linewidth=0.8, alpha=0.9)
        outlier_mark = " | latency outlier" if bool(stim_row.get("late_latency_outlier", False)) else ""
        ax.set_title(
            f"stim {stim_idx} | bin {stim_row['phase_bin']} | phase {stim_row['movement_phase']:.2f} | "
            f"M p2p {stim_row.get('middle_p2p_amplitude', np.nan):.3g} | LR p2p {stim_row.get('late_p2p_amplitude', np.nan):.3g} | "
            f"LR lat {stim_row.get('late_peak_latency_ms', np.nan):.1f} ms{outlier_mark}",
            loc="left", fontsize=8,
        )
        ax.grid(alpha=0.25, linestyle=":")
    axes[-1].set_xlabel("Time after stimulus, ms")
    axes[0].legend(loc="upper right", ncols=3)
    fig.text(0.02, 0.5, "Amplitude, native", rotation="vertical", va="center")
    fig.suptitle("Ten LR-containing epochs: waveform check for phase and latency")
    plt.tight_layout(rect=[0.04, 0.02, 1, 0.96])
    return fig, rank_df


## Load Recording

In [ ]:
if not mat_path.exists():
    raise FileNotFoundError(mat_path)

raw = load_mat_raw(mat_path)
sfreq = float(raw.info["sfreq"])
nyquist = sfreq / 2.0

print("File:", mat_path.name)
print("Raw object:", type(raw).__name__)
print("Sampling frequency, Hz:", sfreq)
print("Nyquist frequency, Hz:", nyquist)
print("Data shape, channels x samples:", (len(raw.ch_names), raw.n_times))
print("Duration, s:", raw.n_times / sfreq)
print("Channels:", raw.ch_names)


## Select Movement Reference Channel

In [ ]:
channel = movement_phase_config["channel"]
if channel not in raw.ch_names:
    raise ValueError(f"Channel {channel!r} is not present. Available channels: {raw.ch_names}")

raw_emg = raw.get_data(picks=[channel])[0]
time_s = np.arange(raw_emg.size) / sfreq

print("Movement reference channel:", channel)
print("raw_emg shape:", raw_emg.shape)
print("raw_emg finite:", bool(np.isfinite(raw_emg).all()))


raw.copy().plot(
    start=0,
    duration=1.0,
    n_channels=len(raw.ch_names),
    scalings={"emg": 5.16987885},
    title=f"Full recording review: {channel} movement reference",
    block=True,
)


## Validate Movement Filter Parameters

In [ ]:
validated_filters = validate_filter_parameters(
    sfreq,
    emg_band=movement_phase_config["emg_band"],
    envelope_lowpass=movement_phase_config["envelope_lowpass"],
    movement_band=movement_phase_config["movement_band"],
)
validated_filters


filter_check_df = pd.DataFrame([validated_filters])
display(filter_check_df)


## Compute Envelope And Movement Phase

In [ ]:
filtered_emg, rectified_emg, _helper_envelope = make_emg_envelope(
    raw_emg,
    sfreq,
    emg_band=movement_phase_config["emg_band"],
    envelope_lowpass=movement_phase_config["envelope_lowpass"],
    filter_order=movement_phase_config["filter_order"],
)

envelope = emg_envelope(
    raw_emg,
    sfreq,
    band=movement_phase_config["emg_band"],
    lowpass=movement_phase_config["envelope_lowpass"],
)

np.testing.assert_allclose(envelope, _helper_envelope, rtol=1e-10, atol=1e-12)

movement_signal = extract_movement_signal(
    envelope,
    sfreq,
    movement_band=movement_phase_config["movement_band"],
    filter_order=movement_phase_config["filter_order"],
)

analytic_signal, hilbert_amplitude, movement_phase = compute_movement_phase(movement_signal)
emg_analytic_signal, emg_hilbert_amplitude, _ = compute_movement_phase(filtered_emg)
cycle_boundaries = detect_cycle_boundaries(movement_phase)

print("filtered_emg shape:", filtered_emg.shape)
print("rectified_emg shape:", rectified_emg.shape)
print("envelope shape:", envelope.shape)
print("movement_signal shape:", movement_signal.shape)
print("emg_hilbert_amplitude shape:", emg_hilbert_amplitude.shape)
print("movement_phase range:", float(np.nanmin(movement_phase)), float(np.nanmax(movement_phase)))
print("cycle boundaries:", len(cycle_boundaries))

boundary_times_s = cycle_boundaries / sfreq
cycle_duration_s = np.diff(cycle_boundaries) / sfreq
cycle_start_times_s = boundary_times_s[:-1]
cycle_low_s, cycle_high_s = cycle_duration_flag_range_s
cycle_outlier_count = int(((cycle_duration_s < cycle_low_s) | (cycle_duration_s > cycle_high_s)).sum())

print("movement cycles:", len(cycle_duration_s))
if len(cycle_duration_s):
    print("cycle duration median, s:", float(np.nanmedian(cycle_duration_s)))
    print("cycle duration mean, s:", float(np.nanmean(cycle_duration_s)))
    print("cycle duration min, s:", float(np.nanmin(cycle_duration_s)))
    print("cycle duration max, s:", float(np.nanmax(cycle_duration_s)))
    print(f"cycles outside {cycle_duration_flag_range_s} s:", cycle_outlier_count)


phase_check_start_s, phase_check_stop_s = movement_phase_config["qc_window_s"]
phase_check_stop_s = min(float(phase_check_stop_s), time_s[-1])
phase_check_mask = (time_s >= phase_check_start_s) & (time_s <= phase_check_stop_s)
phase_check_boundaries = boundary_times_s[(boundary_times_s >= phase_check_start_s) & (boundary_times_s <= phase_check_stop_s)]

fig, axes = plt.subplots(3, 1, figsize=(15, 4.8), sharex=True)
axes[0].plot(time_s[phase_check_mask], raw_emg[phase_check_mask], color="0.70", linewidth=0.55)
axes[0].set_ylabel(f"raw {channel}", rotation=0, ha="right", va="center")
axes[1].plot(time_s[phase_check_mask], envelope[phase_check_mask], color="tab:blue", linewidth=1.1)
axes[1].set_ylabel("|x|+LP5", rotation=0, ha="right", va="center")
axes[2].plot(time_s[phase_check_mask], emg_hilbert_amplitude[phase_check_mask], color="tab:orange", linewidth=0.9)
axes[2].set_ylabel("Hilbert amp", rotation=0, ha="right", va="center")
axes[2].set_xlabel("Time, s")
for ax in axes:
    for boundary_s in phase_check_boundaries:
        ax.axvline(boundary_s, color="tab:red", alpha=0.45, linewidth=0.8)
    ax.grid(alpha=0.18)
fig.suptitle(f"Movement phase QC: {phase_check_start_s:g}-{phase_check_stop_s:g} s")
plt.tight_layout()
plt.show()


fig, axes = plt.subplots(2, 1, figsize=(15, 5.2), sharex=False)
axes[0].plot(time_s, movement_signal, color="tab:cyan", linewidth=0.55)
for boundary_s in boundary_times_s:
    axes[0].axvline(boundary_s, color="tab:red", alpha=0.06, linewidth=0.6)
axes[0].set_title("Full-recording movement signal with detected cycle boundaries")
axes[0].set_ylabel("movement")
axes[0].grid(alpha=0.18)

axes[1].plot(cycle_start_times_s, cycle_duration_s, color="0.25", marker=".", linestyle="-", linewidth=0.7, markersize=3)
if len(cycle_duration_s):
    axes[1].axhline(np.nanmedian(cycle_duration_s), color="tab:blue", linewidth=1.0, label="median")
axes[1].axhspan(cycle_low_s, cycle_high_s, color="tab:green", alpha=0.12, label="expected range")
axes[1].set_title("Detected movement-cycle duration across full recording")
axes[1].set_xlabel("Cycle start time, s")
axes[1].set_ylabel("Duration, s")
axes[1].grid(alpha=0.18)
axes[1].legend(loc="upper right")
plt.tight_layout()
plt.show()


## Assign Movement Phase To Stimuli

In [ ]:
if not manual_annotation_path.exists():
    raise FileNotFoundError(manual_annotation_path)

annotations = mne.read_annotations(manual_annotation_path)
annotation_descriptions = np.asarray(annotations.description, dtype=str)
stim_onsets_s = np.asarray(annotations.onset)[annotation_descriptions == "Stimulus_Auto"]
stim_samples = np.rint(stim_onsets_s * sfreq).astype(int) + raw.first_samp

phase_alignment_summary = validate_phase_alignment(movement_phase, stim_samples, raw, sfreq)
phase_bin_edges, phase_bin_centers = make_phase_bins(n_phase_bins)

stim_phase_dict = assign_phase_to_stimuli(movement_phase, stim_samples, sfreq, first_samp=raw.first_samp)
stim_phases = pd.DataFrame(stim_phase_dict)
stim_phases.insert(0, "stimulus_index", np.arange(len(stim_phases)))
stim_phases = assign_phase_bins(stim_phases, phase_bin_edges)

phase_bin_counts = (
    stim_phases["phase_bin"]
    .value_counts()
    .reindex(phase_bin_labels, fill_value=0)
    .sort_index()
    .rename_axis("phase_bin")
    .rename("stimulus_count")
    .reset_index()
)
small_phase_bins = phase_bin_counts.loc[phase_bin_counts["stimulus_count"] < 0.5 * phase_bin_counts["stimulus_count"].median()]

print("Stimuli:", len(stim_phases))
print("Stimulus sample range:", int(stim_samples.min()), int(stim_samples.max()))
print("Phase bins:", n_phase_bins)
print("Phase bin edges:", np.round(phase_bin_edges, 3))
print("Invalid/missing phase bins:", int(stim_phases["phase_bin"].isna().sum()))
print("Small phase bins (<50% of median count):", small_phase_bins["phase_bin"].tolist())
display(phase_alignment_summary)
display(phase_bin_counts)
display(stim_phases.head())


fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(stim_phases["movement_phase"].dropna(), bins=phase_bin_edges, color="0.45", edgecolor="white")
for edge in phase_bin_edges:
    axes[0].axvline(edge, color="tab:red", alpha=0.25, linewidth=0.8)
axes[0].set_title("Stimulus phases with bin edges")
axes[0].set_xlabel("Movement phase, rad")
axes[0].set_ylabel("Stimuli")
axes[1].bar(phase_bin_counts["phase_bin"], phase_bin_counts["stimulus_count"], color="0.35")
axes[1].set_title("Phase-bin stimulus counts")
axes[1].set_xlabel("Phase bin")
axes[1].set_ylabel("Stimuli")
axes[1].set_xticks(phase_bin_labels)
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## Build Per-Stimulus Response Tables

In [ ]:
if not reviewed_metrics_path.exists():
    raise FileNotFoundError(reviewed_metrics_path)

reviewed_metrics = pd.read_csv(reviewed_metrics_path)
available_metric_columns = [col for col in available_response_metric_names if col in reviewed_metrics.columns]
missing_metric_columns = [col for col in available_response_metric_names if col not in reviewed_metrics.columns]

phase_response_metrics = reviewed_metrics.merge(
    stim_phases[[
        "stimulus_index", "stimulus_sample", "stimulus_time", "movement_phase",
        "phase_bin", "phase_bin_center", "phase_sin", "phase_cos",
    ]],
    on="stimulus_index",
    how="left",
    validate="many_to_one",
)
phase_response_metrics["phase_at_er"] = phase_response_metrics["movement_phase"].where(phase_response_metrics["label"].eq("er"))
phase_response_metrics["phase_at_mr"] = phase_response_metrics["movement_phase"].where(phase_response_metrics["label"].eq("mr"))
phase_response_metrics["phase_at_lr"] = phase_response_metrics["movement_phase"].where(phase_response_metrics["label"].eq("lr"))

response_by_stim = build_response_by_stim(stim_phases, reviewed_metrics, available_metric_columns)
response_by_stim, lr_latency_center_ms, lr_latency_scale_ms = add_lr_latency_outliers(
    response_by_stim,
    value_col="late_peak_latency_ms",
    mad_k=latency_outlier_mad_k,
)
response_by_phase_bin = summarize_by_phase_bin(response_by_stim, available_metric_columns, n_phase_bins)

print("Reviewed response rows:", len(reviewed_metrics))
print("Reviewed stimuli:", reviewed_metrics["stimulus_index"].nunique())
print("Rows with movement phase:", int(phase_response_metrics["movement_phase"].notna().sum()))
print("Available metrics:", available_metric_columns)
print("Missing listed metrics:", missing_metric_columns)
print("LR latency median, ms:", lr_latency_center_ms)
print("LR latency scaled MAD, ms:", lr_latency_scale_ms)
print("LR latency outliers:", int(response_by_stim["late_latency_outlier"].sum()))

response_by_stim.head()


response_presence_by_bin = (
    response_by_stim.groupby("phase_bin")[["early_present", "middle_present", "late_present"]]
    .sum()
    .reindex(phase_bin_labels, fill_value=0)
)
fig, ax = plt.subplots(figsize=(8, 3.5))
response_presence_by_bin.plot(kind="bar", ax=ax, color=["tab:green", "tab:orange", "tab:blue"])
ax.set_title("Reviewed response availability by phase bin")
ax.set_xlabel("Phase bin")
ax.set_ylabel("Stimuli with response")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## Validation Summary

In [ ]:
response_window_summary = (
    reviewed_metrics.groupby("label")
    .agg(
        response_rows=("label", "size"),
        stimulus_count=("stimulus_index", "nunique"),
        start_min_ms=("start_latency_ms", "min"),
        start_median_ms=("start_latency_ms", "median"),
        start_max_ms=("start_latency_ms", "max"),
        peak_median_ms=("peak_latency_ms", "median"),
        end_min_ms=("end_latency_ms", "min"),
        end_median_ms=("end_latency_ms", "median"),
        end_max_ms=("end_latency_ms", "max"),
        duration_median_ms=("duration_ms", "median"),
    )
    .round(4)
)

isi_ms = np.diff(stim_samples) / sfreq * 1000
cycle_duration_s = np.diff(cycle_boundaries) / sfreq
reviewed_end_by_stim = reviewed_metrics.groupby("stimulus_index")["end_latency_ms"].max()
reviewed_common = reviewed_end_by_stim.index[reviewed_end_by_stim.index < len(stim_onsets_s) - 1]
next_isi_for_reviewed_ms = (stim_onsets_s[reviewed_common + 1] - stim_onsets_s[reviewed_common]) * 1000
reviewed_overlap_next = reviewed_end_by_stim.loc[reviewed_common].to_numpy() > next_isi_for_reviewed_ms

timing_delta_s = phase_response_metrics["stimulus_s"] - phase_response_metrics["stimulus_time"]
cycle_low_s, cycle_high_s = cycle_duration_flag_range_s
implausible_cycles = cycle_duration_s[(cycle_duration_s < cycle_low_s) | (cycle_duration_s > cycle_high_s)]

validation_summary = pd.DataFrame([{
    "stimuli_total": len(stim_phases),
    "valid_movement_phase": int(stim_phases["movement_phase"].notna().sum()),
    "excluded_invalid_phase": int(stim_phases["movement_phase"].isna().sum()),
    "phase_bins": n_phase_bins,
    "reviewed_response_rows": len(reviewed_metrics),
    "reviewed_stimuli": int(reviewed_metrics["stimulus_index"].nunique()),
    "median_cycle_duration_s": float(np.nanmedian(cycle_duration_s)),
    "mean_cycle_duration_s": float(np.nanmean(cycle_duration_s)),
    "min_cycle_duration_s": float(np.nanmin(cycle_duration_s)),
    "max_cycle_duration_s": float(np.nanmax(cycle_duration_s)),
    "cycles_flagged_for_inspection": int(len(implausible_cycles)),
    "median_isi_ms": float(np.nanmedian(isi_ms)),
    "mean_isi_ms": float(np.nanmean(isi_ms)),
    "early_duration_median_ms": float(response_window_summary.loc["er", "duration_median_ms"]) if "er" in response_window_summary.index else np.nan,
    "middle_duration_median_ms": float(response_window_summary.loc["mr", "duration_median_ms"]) if "mr" in response_window_summary.index else np.nan,
    "late_duration_median_ms": float(response_window_summary.loc["lr", "duration_median_ms"]) if "lr" in response_window_summary.index else np.nan,
    "late_window_overlaps_next_stimulus": bool(reviewed_overlap_next.any()) if len(reviewed_common) else False,
    "overlapping_reviewed_stimuli": int(reviewed_overlap_next.sum()) if len(reviewed_common) else 0,
    "max_abs_timing_delta_s": float(timing_delta_s.abs().max()),
}])

print("AUC definition currently used: absolute AUC, np.trapezoid(np.abs(segment), time_ms)")
print("RMS definition currently used: sqrt(mean(segment**2))")
print("Latency reference: stimulus onset")
print("Latency note: variation is not automatically artifact; robust outliers are highlighted for waveform inspection.")
display(validation_summary)
display(response_window_summary)
display(response_by_phase_bin)


validation_plot_values = pd.Series({
    "all stimuli": validation_summary.loc[0, "stimuli_total"],
    "reviewed stimuli": validation_summary.loc[0, "reviewed_stimuli"],
    "LR stimuli": int(response_by_stim["late_present"].sum()),
})
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(validation_plot_values.index, validation_plot_values.values, color=["0.45", "tab:orange", "tab:blue"])
ax.set_title("Stimulus and reviewed-response counts")
ax.set_ylabel("Count")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## QC Figure: Raw EMG, Envelope, Movement Signal, And Phase

In [ ]:
qc_start_s, qc_stop_s = movement_phase_config["qc_window_s"]
fig, axes = plot_raw_envelope_phase_segment(
    time_s=time_s,
    raw_emg=raw_emg,
    envelope=envelope,
    movement_signal=movement_signal,
    movement_phase=movement_phase,
    response_by_stim=response_by_stim,
    cycle_boundaries=cycle_boundaries,
    sfreq=sfreq,
    start_s=qc_start_s,
    stop_s=qc_stop_s,
)
plt.show()

print("Interpretation: this confirms that each stimulus is assigned to a meaningful continuous movement phase.")


## QC Figure: Cycle Duration

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(cycle_duration_s, bins=40, color="0.4", alpha=0.85)
ax.axvline(np.nanmedian(cycle_duration_s), color="tab:red", linewidth=1.5, label="median")
ax.axvline(np.nanmean(cycle_duration_s), color="tab:blue", linewidth=1.5, label="mean")
ax.axvspan(0, cycle_duration_flag_range_s[0], color="tab:red", alpha=0.12, label="flag range")
ax.axvspan(cycle_duration_flag_range_s[1], np.nanmax(cycle_duration_s), color="tab:red", alpha=0.12)
ax.set_xlabel("Cycle duration, s")
ax.set_ylabel("Count")
ax.set_title("Movement-cycle duration QC")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

print("Cycle duration median, s:", float(np.nanmedian(cycle_duration_s)))
print("Cycle duration mean, s:", float(np.nanmean(cycle_duration_s)))
print("Cycle duration range, s:", float(np.nanmin(cycle_duration_s)), "to", float(np.nanmax(cycle_duration_s)))
print(f"Cycles flagged for inspection (<{cycle_duration_flag_range_s[0]} s or >{cycle_duration_flag_range_s[1]} s):", len(implausible_cycles))


## Main Figure: Late Response vs M-Response

In [ ]:
fig, axes = plot_lr_vs_m_response(response_by_stim)
plt.show()

print("Interpretation: this shows whether stronger M-responses are associated with stronger late responses, and whether this relationship depends on movement phase.")


## LR Latency Outliers vs M-Response

In [ ]:
fig, ax, lr_latency_plot_df = plot_lr_latency_outliers(
    response_by_stim,
    latency_center_ms=lr_latency_center_ms,
    latency_scale_ms=lr_latency_scale_ms,
    mad_k=latency_outlier_mad_k,
)
plt.show()

latency_outliers = lr_latency_plot_df.loc[lr_latency_plot_df["late_latency_outlier"]].copy()
print("Latency interpretation: latency variation alone is not automatically an artifact. It is suspicious when points are robust outliers, isolated from the main latency cluster, or paired with abnormal waveform/low signal quality.")
print("LR latency outliers shown on graph:", len(latency_outliers))
display(latency_outliers[[
    "stimulus_index", "movement_phase", "phase_bin", "middle_p2p_amplitude",
    "late_p2p_amplitude", "late_peak_latency_ms", "late_latency_robust_z",
]])


## Phase-Binned Summary: M-Response, LR, And Sample Count

In [ ]:
fig, axes = plot_phase_bin_summary(response_by_stim, response_by_phase_bin)
plt.show()

print("M-response amplitude: shows whether the M-response is stable or changes with movement phase.")
print("LR amplitude: shows whether the late response becomes larger in specific movement phases.")
print("LR fraction: shows whether late responses are more likely in specific movement phases.")
print("Stimulus count: shows whether phase-bin comparisons are well supported.")


## Phase At ER, MR, And LR

In [ ]:
fig, axes = plot_phase_at_response_counts(response_by_stim)
plt.show()

phase_at_summary = pd.DataFrame({
    "response": ["ER", "MR", "LR"],
    "responses_present": [
        int(response_by_stim["early_present"].sum()),
        int(response_by_stim["middle_present"].sum()),
        int(response_by_stim["late_present"].sum()),
    ],
    "non_missing_phase_at_response": [
        int(response_by_stim["phase_at_er"].notna().sum()),
        int(response_by_stim["phase_at_mr"].notna().sum()),
        int(response_by_stim["phase_at_lr"].notna().sum()),
    ],
})
print("Interpretation: this shows which movement phases contain stimuli with ER, MR, and LR responses.")
display(phase_at_summary)


## Ten Epoch Examples

In [ ]:
best_lr_epoch_figure, best_lr_epochs_by_phase = plot_10_epochs_by_phase(
    response_by_stim,
    phase_response_metrics,
    raw_emg,
    time_s,
    sfreq,
    n_plots=10,
)
if best_lr_epoch_figure is not None:
    plt.show()
    display(best_lr_epochs_by_phase[[
        "stimulus_index", "stimulus_time_s", "movement_phase", "phase_bin",
        "middle_p2p_amplitude", "late_p2p_amplitude", "late_peak_latency_ms",
        "late_latency_outlier", "late_count",
    ]])
else:
    print("No LR-containing epochs available for the 10-epoch QC plot.")

print("Interpretation: this verifies that dataframe relationships correspond to real visible waveforms.")


## Metric Availability Summary

In [ ]:
metric_availability = []
for metric_col, description in m_response_metrics + lr_response_metrics:
    metric_availability.append({
        "metric": metric_col,
        "description": description,
        "available_values": int(pd.to_numeric(response_by_stim.get(metric_col, pd.Series(dtype=float)), errors="coerce").notna().sum()),
    })
metric_availability = pd.DataFrame(metric_availability)
display(metric_availability)

display(
    phase_response_metrics.groupby("label")[available_metric_columns]
    .agg(["count", "median"])
    .round(4)
)


## Notes: movement phase is assigned at stimulus onset; inspect QC plots before using the summary tables.


## Final Objects

In [ ]:
print("response_by_stim shape:", response_by_stim.shape)
print("response_by_phase_bin shape:", response_by_phase_bin.shape)
display(response_by_stim.head())
display(response_by_phase_bin.head())


## Final Interactive Response Review


In [ ]:
open_final_interactive_review = True

if open_final_interactive_review:
    if not final_annotation_path.exists():
        raise FileNotFoundError(final_annotation_path)
    mne.viz.set_browser_backend("qt")
    review_raw = raw.copy()
    review_raw.set_annotations(mne.read_annotations(final_annotation_path))
    print("Opening final interactive response review. Close the browser to continue.")
    review_raw.plot(
        start=movement_phase_config["qc_window_s"][0],
        duration=10,
        n_channels=len(review_raw.ch_names),
        scalings={"emg": 8.0779357 / 1.5},
        title="Final response review: stimuli with ER/MR/LR annotations",
        block=True,
    )
else:
    print("Final interactive review skipped. Set open_final_interactive_review = True to open it.")
